In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/video-inpainting-small-dataset/humans/vid6_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/humans/vid5_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/humans/vid4_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/humans/vid1_540x960_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/humans/vid2_540x960_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/humans/vid3_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cars/vid3_540x960_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cars/vid6_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cars/vid5_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cars/vid4_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cars/vid2_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cars/vid1_960x540_30fps_10s.mp4
/kaggle/input/video-inpainting-small-dataset/cats/vid3_960x540_3

# Standardisation method (already applied before upload) # 

All videos in this dataset (cats / cars / humans) were standardized offline using PowerShell + FFmpeg to ensure consistent resolution, frame rate, encoding, and (for cars/humans) consistent clip length.

## Common settings (all categories) ##
* **Frame rate:** forced to 30 fps
* **Aspect ratio handling:**  original aspect ratio preserved; video is resized to fit within a target canvas, then letterboxed/padded to fill the canvas (no stretching)
* **Video codec:** H.264 (libx264) with -crf 18 and -preset slow
* **Pixel format:** yuv420p (widest compatibility)
* **Audio:** removed (-an)
* **Sample aspect ratio:** normalized to 1:1 (setsar=1) for consistency across sources

## Canvas size (depends on orientation / category) ##
* Cat videos were standardized to a single landscape canvas: 960×540
* Cars and humans videos are more diverse (including portrait and ultrawide). To avoid shrinking portrait subjects into a landscape frame, we used an orientation-aware target:
    * Landscape videos: 960×540
    * Portrait videos: 540×960

All clips were also standardized to a fixed duration of 10 seconds.

# Overview #

Use this notebook to extract frames from standardized cat videos stored in a Kaggle input dataset. It supports:

* Extracting frames as JPG (recommended for size) or PNG
* Optional frame stride (e.g., keep every 2nd frame)
* Optional ZIP per video (helps with file-count limits and easier downloading)
* A manifest CSV per video (frame index, timestamp, path)

In [2]:
from pathlib import Path
import re
import json
import shutil
import subprocess
from datetime import datetime

# Settings #

In [4]:
from pathlib import Path

# -----------------------------
# INPUT DATASET STRUCTURE
# -----------------------------
# /kaggle/input/video-inpainting-small-dataset/
#   cats/*.mp4
#   cars/*.mp4
#   humans/*.mp4
INPUT_ROOT = Path("/kaggle/input/video-inpainting-small-dataset")
CATEGORIES = ["cats", "cars", "humans"]   # process all in one run

# -----------------------------
# OUTPUT
# -----------------------------
OUT_ROOT = Path("/kaggle/working/video_inpainting_frames_dataset")

# We’ll store outputs like:
# OUT_ROOT/
#   frames/<category>/<video_id>/<frame>.jpg
#   manifests/<category>/<video_id>.csv
#   zips/<category>/<video_id>.zip
OUT_FRAMES_DIR = OUT_ROOT / "frames"
OUT_MANIFESTS_DIR = OUT_ROOT / "manifests"
OUT_ZIPS_DIR = OUT_ROOT / "zips"

# Also write a master index across all videos:
OUT_MASTER_INDEX = OUT_ROOT / "index_videos.csv"

# -----------------------------
# EXTRACTION SETTINGS
# -----------------------------
OUTPUT_FORMAT = "jpg"   # "jpg" or "png"
JPG_QUALITY = 2         # FFmpeg -q:v (lower is better quality, larger files). Typical: 2-5.
PNG_COMPRESSION = 6     # FFmpeg -compression_level 0-9 (higher smaller, slower). Only used if OUTPUT_FORMAT="png".

FRAME_STRIDE = 1        # 1 = keep every frame, 2 = every other frame, etc.
LIMIT_VIDEOS_PER_CATEGORY = None  # e.g., 2 for testing; None for all
OVERWRITE = False       # if False, skip videos whose output already exists

# -----------------------------
# PACKAGING SETTINGS
# -----------------------------
ZIP_EACH_VIDEO = True
DELETE_RAW_FRAMES_AFTER_ZIP = False  # True to save space

# -----------------------------
# LOGGING
# -----------------------------
VERBOSE = True

# Helper functions (FFmpeg extract + zip + manifests) #

In [5]:
import csv
import shutil
import subprocess
from zipfile import ZipFile, ZIP_DEFLATED

def log(msg: str):
    if VERBOSE:
        print(msg)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def run(cmd):
    # Show command only if verbose
    if VERBOSE:
        print(" ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

def extract_frames_ffmpeg(video_path: Path, frames_dir: Path):
    """
    Extract frames to frames_dir/%05d.<ext>
    Uses stride via select filter so you can keep exact frame numbers at 30fps sources.
    """
    ensure_dir(frames_dir)
    out_pattern = str(frames_dir / f"%05d.{OUTPUT_FORMAT}")

    # select every Nth frame: keep when n % stride == 0
    # Need to escape commas for ffmpeg expression on some shells; in Python list args, we pass as-is.
    if FRAME_STRIDE == 1:
        vf = None
    else:
        vf = f"select=not(mod(n\\,{FRAME_STRIDE}))"

    cmd = ["ffmpeg", "-y" if OVERWRITE else "-n", "-i", str(video_path), "-vsync", "vfr"]
    if vf is not None:
        cmd += ["-vf", vf]

    if OUTPUT_FORMAT.lower() == "jpg":
        cmd += ["-q:v", str(JPG_QUALITY)]
    elif OUTPUT_FORMAT.lower() == "png":
        cmd += ["-compression_level", str(PNG_COMPRESSION)]
    else:
        raise ValueError("OUTPUT_FORMAT must be 'jpg' or 'png'")

    cmd += [out_pattern]
    run(cmd)

def list_frames(frames_dir: Path):
    frames = sorted(frames_dir.glob(f"*.{OUTPUT_FORMAT}"))
    return frames

def write_video_manifest_csv(manifest_path: Path, category: str, video_id: str, frames):
    """
    Per-video manifest listing all extracted frames.
    """
    ensure_dir(manifest_path.parent)
    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["category", "video_id", "frame_index", "frame_file"])
        for i, fp in enumerate(frames):
            w.writerow([category, video_id, i, str(fp)])

def zip_frames(frames_dir: Path, zip_path: Path):
    ensure_dir(zip_path.parent)
    frames = list_frames(frames_dir)
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as z:
        for fp in frames:
            # Store inside the zip as just the filename (00001.jpg etc.)
            z.write(fp, arcname=fp.name)

def safe_video_id(category: str, video_path: Path) -> str:
    # Prefix category to avoid collisions if two categories share same basename
    return f"{category}__{video_path.stem}"

# Main loop (process all categories + write a master index) #

In [6]:
ensure_dir(OUT_ROOT)
ensure_dir(OUT_FRAMES_DIR)
ensure_dir(OUT_MANIFESTS_DIR)
ensure_dir(OUT_ZIPS_DIR)

master_rows = []
total_videos = 0

for category in CATEGORIES:
    videos_dir = INPUT_ROOT / category
    if not videos_dir.exists():
        log(f"[WARN] Missing category folder: {videos_dir}")
        continue

    videos = sorted(videos_dir.glob("*.mp4"))
    if LIMIT_VIDEOS_PER_CATEGORY is not None:
        videos = videos[:LIMIT_VIDEOS_PER_CATEGORY]

    log(f"\n=== Category: {category} | videos found: {len(videos)} ===")

    for video_path in videos:
        total_videos += 1
        video_id = safe_video_id(category, video_path)

        frames_dir = OUT_FRAMES_DIR / category / video_id
        manifest_path = OUT_MANIFESTS_DIR / category / f"{video_id}.csv"
        zip_path = OUT_ZIPS_DIR / category / f"{video_id}.zip"

        # Skip if already done (you can choose your own skip condition)
        if not OVERWRITE:
            if ZIP_EACH_VIDEO and zip_path.exists():
                log(f"[SKIP] {video_id} (zip exists)")
                master_rows.append([category, video_id, str(video_path), "", str(zip_path), True])
                continue
            if (not ZIP_EACH_VIDEO) and frames_dir.exists() and any(frames_dir.glob(f"*.{OUTPUT_FORMAT}")):
                log(f"[SKIP] {video_id} (frames exist)")
                master_rows.append([category, video_id, str(video_path), str(frames_dir), "", True])
                continue

        log(f"[RUN] {video_id} <- {video_path.name}")

        # Clean existing frames folder if overwriting
        if OVERWRITE and frames_dir.exists():
            shutil.rmtree(frames_dir)

        extract_frames_ffmpeg(video_path, frames_dir)
        frames = list_frames(frames_dir)
        write_video_manifest_csv(manifest_path, category, video_id, frames)

        if ZIP_EACH_VIDEO:
            if OVERWRITE and zip_path.exists():
                zip_path.unlink()
            zip_frames(frames_dir, zip_path)
            if DELETE_RAW_FRAMES_AFTER_ZIP:
                shutil.rmtree(frames_dir)

        master_rows.append([
            category,
            video_id,
            str(video_path),
            str(frames_dir) if (not DELETE_RAW_FRAMES_AFTER_ZIP or not ZIP_EACH_VIDEO) else "",
            str(zip_path) if ZIP_EACH_VIDEO else "",
            False
        ])

log(f"\nDone. Total videos processed/checked: {total_videos}")

# Write master index (one row per video)
with OUT_MASTER_INDEX.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["category", "video_id", "source_video", "frames_dir", "zip_path", "skipped"])
    w.writerows(master_rows)

print(f"Master index written to: {OUT_MASTER_INDEX}")
print(f"Outputs root: {OUT_ROOT}")


=== Category: cats | videos found: 6 ===
[RUN] cats__vid1_960x540_30fps <- vid1_960x540_30fps.mp4
ffmpeg -n -i /kaggle/input/video-inpainting-small-dataset/cats/vid1_960x540_30fps.mp4 -vsync vfr -q:v 2 /kaggle/working/video_inpainting_frames_dataset/frames/cats/cats__vid1_960x540_30fps/%05d.jpg


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

[RUN] cats__vid2_960x540_30fps <- vid2_960x540_30fps.mp4
ffmpeg -n -i /kaggle/input/video-inpainting-small-dataset/cats/vid2_960x540_30fps.mp4 -vsync vfr -q:v 2 /kaggle/working/video_inpainting_frames_dataset/frames/cats/cats__vid2_960x540_30fps/%05d.jpg


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

[RUN] cats__vid3_960x540_30fps <- vid3_960x540_30fps.mp4
ffmpeg -n -i /kaggle/input/video-inpainting-small-dataset/cats/vid3_960x540_30fps.mp4 -vsync vfr -q:v 2 /kaggle/working/video_inpainting_frames_dataset/frames/cats/cats__vid3_960x540_30fps/%05d.jpg


...

[RUN] humans__vid5_960x540_30fps_10s <- vid5_960x540_30fps_10s.mp4
ffmpeg -n -i /kaggle/input/video-inpainting-small-dataset/humans/vid5_960x540_30fps_10s.mp4 -vsync vfr -q:v 2 /kaggle/working/video_inpainting_frames_dataset/frames/humans/humans__vid5_960x540_30fps_10s/%05d.jpg


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

[RUN] humans__vid6_960x540_30fps_10s <- vid6_960x540_30fps_10s.mp4
ffmpeg -n -i /kaggle/input/video-inpainting-small-dataset/humans/vid6_960x540_30fps_10s.mp4 -vsync vfr -q:v 2 /kaggle/working/video_inpainting_frames_dataset/frames/humans/humans__vid6_960x540_30fps_10s/%05d.jpg


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab


Done. Total videos processed/checked: 18
Master index written to: /kaggle/working/video_inpainting_frames_dataset/index_videos.csv
Outputs root: /kaggle/working/video_inpainting_frames_dataset


# How to publish as a Kaggle dataset #

1. Run the notebook to completion.
2. Open the right panel → Output and confirm files exist under:
    * /kaggle/working/video_inpainting_frames_dataset/

Expected structure (depending on the settings):
* frames/\<category\>/\<video_id\>/*.(jpg|png) (if not deleted after zipping)
* zips/\<category\>/\<video_id\>.zip (if ZIP_EACH_VIDEO=True)
* manifests/\<category\>/\<video_id\>.csv
* index_videos.csv (master index across cats/cars/humans)

3. Click Save Version (top right).
4. In the save dialog, ensure “Save output” is enabled.

Kaggle will create a new dataset version containing the notebook outputs (frames and/or ZIPs and manifests).